# 04 · Derivadas, gradientes y descenso, desde cero

**Módulo 1 · Sesión 3** — Fundamentos matemáticos

## Objetivos

Casi todo modelo de Machine Learning se entrena minimizando una función de error. Este
notebook construye la maquinaria que hace posible esa minimización:

1. La derivada como pendiente, calculada numérica y analíticamente.
2. El gradiente: la derivada cuando hay varias variables.
3. **Descenso del gradiente**: el algoritmo que entrena casi todo.
4. El papel de la tasa de aprendizaje (y cómo se rompe todo si se elige mal).
5. La regla de la cadena, que es literalmente la retropropagación de la sesión 13.

Al final entrenaremos una regresión sin usar scikit-learn, solo con lo construido aquí.
Es el puente directo hacia la sesión 6.

## Paquetes

`numpy`, `pandas`, `matplotlib`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-1-fundamentos-ciclo-vida/notebooks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMILLA = 42

## 1. La derivada es una pendiente

La derivada de una función en un punto responde: **si me muevo un poquito a la derecha,
¿cuánto sube o baja la función?**

$$
f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}
$$

Sin el límite, esa fórmula es directamente calculable. Eso se llama derivada numérica.

In [ ]:
def f(x):
    """Función de ejemplo: una parábola con mínimo en x = 3."""
    return (x - 3) ** 2 + 2


def derivada_numerica(funcion, x, h=1e-6):
    """Aproximación por diferencias centradas: más precisa que (f(x+h)-f(x))/h."""
    return (funcion(x + h) - funcion(x - h)) / (2 * h)


def derivada_analitica(x):
    """Derivada exacta de f: d/dx[(x-3)^2 + 2] = 2(x-3)."""
    return 2 * (x - 3)


for x in [0.0, 2.0, 3.0, 5.0]:
    print(
        f"x = {x:4.1f}   f(x) = {f(x):6.2f}   "
        f"numérica = {derivada_numerica(f, x):8.4f}   "
        f"analítica = {derivada_analitica(x):8.4f}"
    )

Fíjate en el signo, porque es la clave de todo:

- En $x=0$ la derivada es **negativa**: la función baja hacia la derecha → hay que avanzar.
- En $x=5$ es **positiva**: la función sube hacia la derecha → hay que retroceder.
- En $x=3$ es **cero**: estamos en el mínimo.

> **La regla que gobierna todo el entrenamiento:** para minimizar, muévete en dirección
> **contraria** a la derivada.

In [ ]:
x = np.linspace(-1, 7, 200)
fig, eje = plt.subplots(figsize=(7, 4.5))
eje.plot(x, f(x), label="$f(x) = (x-3)^2 + 2$")

for punto, color in [(0.0, "tab:red"), (5.0, "tab:green")]:
    pendiente = derivada_analitica(punto)
    recta = f(punto) + pendiente * (x - punto)
    eje.plot(x, recta, "--", color=color, linewidth=1,
             label=f"tangente en x={punto:.0f} (pendiente {pendiente:.0f})")
    eje.plot(punto, f(punto), "o", color=color)

eje.plot(3, f(3), "k*", markersize=14, label="mínimo")
eje.set_ylim(0, 20)
eje.set_xlabel("x")
eje.set_ylabel("f(x)")
eje.legend(fontsize=8)
eje.set_title("La derivada es la pendiente de la tangente")
plt.tight_layout()
plt.show()

## 2. Descenso del gradiente en una dimensión

El algoritmo completo cabe en una línea. Partimos de un punto cualquiera y repetimos:

$$
x \leftarrow x - \eta \, f'(x)
$$

donde $\eta$ (eta) es la **tasa de aprendizaje** (*learning rate*): cuánto avanzamos en
cada paso.

In [ ]:
def descenso_1d(derivada, x_inicial, tasa, pasos=25):
    """Devuelve la trayectoria completa del descenso, para poder graficarla."""
    x = x_inicial
    trayectoria = [x]
    for _ in range(pasos):
        x = x - tasa * derivada(x)
        trayectoria.append(x)
    return np.array(trayectoria)


trayectoria = descenso_1d(derivada_analitica, x_inicial=0.0, tasa=0.1)

print(f"{'paso':>5} {'x':>9} {'f(x)':>9} {'derivada':>10}")
for paso in [0, 1, 2, 5, 10, 25]:
    xi = trayectoria[paso]
    print(f"{paso:>5} {xi:>9.4f} {f(xi):>9.4f} {derivada_analitica(xi):>10.4f}")

print(f"\nMínimo real: x = 3.0000")
print(f"Encontrado:  x = {trayectoria[-1]:.4f}")

Sin saber nada de la función salvo cómo calcular su derivada, el algoritmo encontró el
mínimo. **Eso es entrenar un modelo.**

## 3. La tasa de aprendizaje lo decide todo

Es el hiperparámetro más delicado. Tres regímenes:

In [ ]:
casos = [
    (0.01, "Muy pequeña: converge, pero lentísimo"),
    (0.10, "Adecuada: converge en pocos pasos"),
    (0.90, "Grande: oscila, aunque todavía converge"),
    (1.05, "Demasiado grande: diverge"),
]

fig, ejes = plt.subplots(1, 4, figsize=(16, 3.8))
rejilla = np.linspace(-2, 8, 200)

for eje, (tasa, titulo) in zip(ejes, casos):
    tr = descenso_1d(derivada_analitica, x_inicial=0.0, tasa=tasa, pasos=20)
    eje.plot(rejilla, f(rejilla), color="lightgray", linewidth=2)
    eje.plot(tr, f(tr), "o-", markersize=4, linewidth=1)
    eje.set_title(f"$\\eta$ = {tasa}\n{titulo}", fontsize=9)
    eje.set_ylim(0, 30)
    eje.set_xlim(-2, 8)

plt.tight_layout()
plt.show()

print(f"{'eta':>6} {'x tras 20 pasos':>18} {'x tras 80 pasos':>18}")
for tasa, _ in casos:
    corto = descenso_1d(derivada_analitica, 0.0, tasa, pasos=20)[-1]
    largo = descenso_1d(derivada_analitica, 0.0, tasa, pasos=80)[-1]
    fmt = lambda v: "diverge" if not np.isfinite(v) or abs(v) > 1e3 else f"{v:.4f}"
    print(f"{tasa:>6} {fmt(corto):>18} {fmt(largo):>18}")

- **Demasiado pequeña**: converge, pero puede tardar más de lo que estás dispuesto a
  esperar. Con $\eta = 0.01$, 20 pasos no bastan ni de lejos.
- **Adecuada**: llega rápido y se queda.
- **Demasiado grande**: cada paso se pasa de largo. Con $\eta = 1.05$ el error no se ve en
  20 pasos, pero a los 80 ya se ha ido a pique. **Una tasa mala puede tardar en delatarse.**

> **Un detalle que no es casualidad.** Con $\eta = 0.1$ y con $\eta = 0.9$ se obtiene
> *exactamente* el mismo valor. Para esta función el paso equivale a
> $(x - 3) \leftarrow (1 - 2\eta)(x - 3)$: con $\eta = 0.1$ el factor es $0.8$ y con
> $\eta = 0.9$ es $-0.8$. El mismo tamaño, signo alterno — por eso una desciende suave y la
> otra rebota de lado a lado, pero tras un número **par** de pasos coinciden. La
> convergencia depende de $|1 - 2\eta| < 1$, es decir $\eta < 1$; en $\eta = 1.05$ el factor
> es $-1.1$ y cada paso amplifica el error un 10 %.

Cuando en la sesión 13 entrenes una red neuronal y la pérdida se vuelva `NaN`, este será el
primer sospechoso.

## 4. Gradiente: varias variables a la vez

Con más de una variable, la derivada se convierte en el **gradiente**: el vector de
derivadas parciales, una por variable.

$$
\nabla f = \left[\frac{\partial f}{\partial x_1},\ \frac{\partial f}{\partial x_2}\right]
$$

El gradiente apunta en la dirección de **máximo ascenso**. Por eso descendemos restándolo.

In [ ]:
def g(punto):
    """Paraboloide con mínimo en (2, -1)."""
    x, y = punto
    return (x - 2) ** 2 + 3 * (y + 1) ** 2


def gradiente_g(punto):
    x, y = punto
    return np.array([2 * (x - 2), 6 * (y + 1)])


punto = np.array([0.0, 0.0])
print(f"En {punto}:")
print(f"  g       = {g(punto):.2f}")
print(f"  ∇g      = {gradiente_g(punto)}")
print(f"  dirección de descenso = {-gradiente_g(punto)}")

El descenso funciona igual, solo que ahora movemos un vector.

In [ ]:
def descenso_2d(gradiente, inicial, tasa, pasos=40):
    p = np.array(inicial, dtype=float)
    trayectoria = [p.copy()]
    for _ in range(pasos):
        p = p - tasa * gradiente(p)
        trayectoria.append(p.copy())
    return np.array(trayectoria)


ruta = descenso_2d(gradiente_g, [0.0, 0.0], tasa=0.1)
print(f"Inicio:      {ruta[0]}")
print(f"Final:       {ruta[-1].round(4)}")
print(f"Mínimo real: [2. -1.]")

In [ ]:
malla_x, malla_y = np.meshgrid(np.linspace(-1, 5, 100), np.linspace(-3, 2, 100))
malla_z = (malla_x - 2) ** 2 + 3 * (malla_y + 1) ** 2

fig, eje = plt.subplots(figsize=(6.5, 5))
contorno = eje.contour(malla_x, malla_y, malla_z, levels=25, cmap="Blues_r", linewidths=0.8)
eje.plot(ruta[:, 0], ruta[:, 1], "o-", color="tab:red", markersize=3, linewidth=1,
         label="trayectoria del descenso")
eje.plot(2, -1, "k*", markersize=15, label="mínimo")
eje.set_xlabel("$x_1$")
eje.set_ylabel("$x_2$")
eje.set_title("Descenso del gradiente sobre las curvas de nivel")
eje.legend()
plt.tight_layout()
plt.show()

La trayectoria cruza las curvas de nivel **perpendicularmente**: esa es la propiedad
geométrica del gradiente. Y baja más rápido en la dirección donde la superficie es más
empinada (aquí, el eje $x_2$, porque su coeficiente es 3 y no 1).

> **Conexión con el escalado.** Si una variable tiene una escala mucho mayor que otra, las
> curvas de nivel se vuelven elipses muy alargadas y el descenso zigzaguea sin avanzar.
> Es otra razón —ahora de optimización— para estandarizar las variables.

## 5. La regla de la cadena

Cuando una función está compuesta de otras, su derivada es el **producto** de las derivadas
de cada eslabón:

$$
\frac{d}{dx}f(g(x)) = f'(g(x)) \cdot g'(x)
$$

Verifiquémoslo con $h(x) = (2x + 1)^3$, que es $f(u)=u^3$ compuesta con $u=g(x)=2x+1$.

In [ ]:
def h(x):
    return (2 * x + 1) ** 3


def derivada_cadena(x):
    u = 2 * x + 1
    df_du = 3 * u**2   # derivada de la capa externa
    du_dx = 2          # derivada de la capa interna
    return df_du * du_dx


for x in [0.0, 1.0, 2.0]:
    print(
        f"x = {x:.1f}   regla de la cadena = {derivada_cadena(x):10.4f}   "
        f"numérica = {derivada_numerica(h, x):10.4f}"
    )

> **Esto es la retropropagación.** Una red neuronal es una composición de funciones, una
> por capa. Para saber cuánto contribuye un peso de la primera capa al error final, se
> multiplican las derivadas de todas las capas que hay en medio, de atrás hacia adelante.
> La sesión 13 le pone nombre y notación, pero la matemática es exactamente esta.

## 6. Entrenar una regresión sin scikit-learn

Juntemos todo. Ajustaremos una recta a datos reales del módulo minimizando el **error
cuadrático medio**:

$$
\mathcal{L}(\beta_0, \beta_1) = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - (\beta_0 + \beta_1 x_i)\right)^2
$$

Sus derivadas parciales salen de la regla de la cadena:

$$
\frac{\partial \mathcal{L}}{\partial \beta_0} = -\frac{2}{n}\sum_i (y_i - \hat{y}_i)
\qquad
\frac{\partial \mathcal{L}}{\partial \beta_1} = -\frac{2}{n}\sum_i (y_i - \hat{y}_i)\,x_i
$$

In [ ]:
datos = pd.read_csv("../datos/rendimiento-estudiantes.csv")

x_datos = datos["promedio_anterior"].to_numpy()
y_datos = datos["nota_final"].to_numpy()

# Estandarizamos x para que el descenso se comporte bien (sección 4).
x_media, x_desv = x_datos.mean(), x_datos.std()
x_z = (x_datos - x_media) / x_desv

print(f"n = {len(x_datos)} estudiantes")
print(f"promedio_anterior: media {x_media:.2f}, desviación {x_desv:.2f}")

In [ ]:
def perdida(beta_0, beta_1, x, y):
    residuales = y - (beta_0 + beta_1 * x)
    return np.mean(residuales**2)


def gradiente_perdida(beta_0, beta_1, x, y):
    residuales = y - (beta_0 + beta_1 * x)
    d_beta_0 = -2 * np.mean(residuales)
    d_beta_1 = -2 * np.mean(residuales * x)
    return d_beta_0, d_beta_1


beta_0, beta_1 = 0.0, 0.0
tasa = 0.1
historial = []

for paso in range(200):
    historial.append(perdida(beta_0, beta_1, x_z, y_datos))
    g0, g1 = gradiente_perdida(beta_0, beta_1, x_z, y_datos)
    beta_0 -= tasa * g0
    beta_1 -= tasa * g1

print(f"Pérdida inicial: {historial[0]:.4f}")
print(f"Pérdida final:   {historial[-1]:.4f}")
print(f"\nCoeficientes (escala estandarizada): b0 = {beta_0:.4f}, b1 = {beta_1:.4f}")

### ¿Acertó?

Este problema tiene solución exacta (mínimos cuadrados, sesión 6). Comparemos.

In [ ]:
b1_exacto = np.sum((x_z - x_z.mean()) * (y_datos - y_datos.mean())) / np.sum((x_z - x_z.mean()) ** 2)
b0_exacto = y_datos.mean() - b1_exacto * x_z.mean()

comparacion = pd.DataFrame(
    {
        "coeficiente": ["beta_0", "beta_1"],
        "descenso": [round(beta_0, 4), round(beta_1, 4)],
        "solución exacta": [round(b0_exacto, 4), round(b1_exacto, 4)],
    }
)
print(comparacion.to_string(index=False))

# Volvemos a la escala original para interpretar.
pendiente_original = beta_1 / x_desv
intercepto_original = beta_0 - beta_1 * x_media / x_desv
print(f"\nEn unidades originales: nota = {intercepto_original:.3f} + "
      f"{pendiente_original:.3f} x promedio_anterior")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))

ejes[0].plot(historial)
ejes[0].set_xlabel("Paso")
ejes[0].set_ylabel("Error cuadrático medio")
ejes[0].set_title("Curva de aprendizaje")
ejes[0].set_yscale("log")

ejes[1].scatter(x_datos, y_datos, alpha=0.35, edgecolor="none", label="estudiantes")
rejilla_x = np.linspace(x_datos.min(), x_datos.max(), 100)
rejilla_z = (rejilla_x - x_media) / x_desv
ejes[1].plot(rejilla_x, beta_0 + beta_1 * rejilla_z, "r-", linewidth=2, label="recta ajustada")
ejes[1].set_xlabel("Promedio anterior")
ejes[1].set_ylabel("Nota final")
ejes[1].set_title("Resultado del descenso del gradiente")
ejes[1].legend()

plt.tight_layout()
plt.show()

La curva de aprendizaje (en escala logarítmica) cae rápido y se aplana: señal de
convergencia. Cuando entrenes modelos de verdad, esta gráfica será tu principal instrumento
de diagnóstico.

## Resumen

| Concepto | Idea | Dónde reaparece |
|---|---|---|
| Derivada | Pendiente: hacia dónde crece la función | Toda optimización |
| Descenso del gradiente | Moverse en contra de la derivada | Regresión (S6), redes (S13) |
| Tasa de aprendizaje | Tamaño del paso; muy grande diverge | S6, S13 |
| Gradiente | Derivadas parciales, una por parámetro | Todo modelo con más de un parámetro |
| Regla de la cadena | Derivada de una composición = producto | Retropropagación (S13) |
| Función de pérdida | Lo que el entrenamiento minimiza | Todo el curso |

Lo que acabas de escribir a mano es, en esencia, lo que `LinearRegression().fit()` hace por
ti. La diferencia es que ahora sabes qué pasa por dentro cuando algo falla.

## Para practicar

1. Cambia la tasa a 0.5 y a 1.5. ¿En qué punto deja de converger? Relaciónalo con la
   sección 3.
2. Entrena **sin estandarizar** `x` (usa `x_datos` en lugar de `x_z`) con tasa 0.1. ¿Qué
   ocurre y por qué?
3. Extiende el descenso a dos variables predictoras (`promedio_anterior` y
   `horas_estudio_semana`). Necesitarás un gradiente con tres componentes.
4. Implementa un criterio de parada: detener cuando la pérdida mejore menos de $10^{-8}$
   entre pasos. ¿Cuántos pasos hacían falta realmente?